In [8]:
!pip install pytest testbook ipykernel

In [10]:
import pytest
import json
from testbook import testbook

NOTEBOOK_PATH = 'agent_website.ipynb'

def test_agent_pipeline(tb):
    tb.inject("""
        import torch
        from unittest.mock import MagicMock
        import requests

        fake_output = torch.randn(1, len(label_map))
        net_best2.forward = MagicMock(return_value=fake_output)
        
        mock_resp = MagicMock()
        mock_resp.json.return_value = {
            "answer_box": {"answer": "Mocked Fact"},
            "organic_results": [{"snippet": "Mocked Evidence"}]
        }
        requests.get = MagicMock(return_value=mock_resp)
    """)

    func_political_bias = tb.ref("func_political_bias")
    func_spam = tb.ref("func_spam")
    func_sensationalism = tb.ref("func_sensationalism")
    func_BERT = tb.ref("func_BERT")
    func_web_search = tb.ref("func_web_search")
    
    test_text = "The government announced a 5% increase in spending."

    bias_res = json.loads(func_political_bias(test_text))
    assert "stat_density" in bias_res
    
    spam_res = json.loads(func_spam("Buy cheap watches!"))
    assert 0.0 <= spam_res["spam_probability"] <= 1.0
    
    sensation_res = json.loads(func_sensationalism("Shocking disaster!"))
    assert sensation_res["emotional_intensity"] > 0

    bert_res = json.loads(func_BERT(test_text))
    assert "model_prediction" in bert_res
    assert "class_probabilities" in bert_res

    search_res = func_web_search("What is the capital of France?")
    assert "Mocked Fact" in search_res

try:
    with testbook(NOTEBOOK_PATH, execute=True) as tb:
        test_agent_pipeline(tb)
        print("Unit tests passed")
except Exception as e:
    print(f"Tests failed: {e}")

Unit tests passed
